# CA4 Question 2 — S&P 500 Forecasting

Baseline MLP and recursive CNN+LSTM / CNN+GRU architectures with leakage-safe splits, stationarity analysis, and comparison table.

In [ ]:
# Install required packages if not available
try:
    import yfinance as yf
except ImportError:
    print("Installing yfinance...")
    !pip install yfinance

try:
    import statsmodels.api as sm
except ImportError:
    print("Installing statsmodels...")
    !pip install statsmodels

import os
import sys
import time
import math
import random
from pathlib import Path
from typing import Tuple, List

# Install required packages if not available
required_packages = ['yfinance', 'statsmodels']
for pkg in required_packages:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f"Installing {pkg}...")
        !{sys.executable} -m pip install {pkg}

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import yfinance as yf
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# Device selection
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Device: {device}")
print(f"Torch: {torch.__version__}")

# Add src to path (relative to notebook cwd)
proj_root = Path.cwd().parent.parent  # Go up to CA4 root
src_path = proj_root / "codes" / "src"
if src_path.exists():
    sys.path.insert(0, str(src_path))
    print(f"Added src: {src_path}")
else:
    print(f"⚠ src path not found: {src_path}")

In [ ]:
# 2. Fetch S&P 500 data (2000–2025)
raw_path = proj_root / "data_sp500_raw.csv"
if raw_path.exists():
    df_raw = pd.read_csv(raw_path, parse_dates=['Date'], index_col='Date')
    print("Loaded data from cache")
else:
    try:
        df_raw = yf.download("^GSPC", start="2000-01-01", auto_adjust=False)
        df_raw.index.name = "Date"
        df_raw.to_csv(raw_path)
        print("Downloaded data from Yahoo Finance")
    except Exception as e:
        print(f"Failed to download data: {e}")
        print("Please ensure yfinance is installed and you have internet connection")
        # Create sample data for demonstration
        dates = pd.date_range('2000-01-01', '2024-12-31', freq='D')
        np.random.seed(42)
        prices = 1000 + np.cumsum(np.random.randn(len(dates)) * 5)
        df_raw = pd.DataFrame({'Open': prices, 'High': prices * 1.01, 'Low': prices * 0.99, 'Close': prices}, index=dates)
        print("Using synthetic data for demonstration")

print(df_raw.head())
print(df_raw.tail())
print(df_raw.isna().sum())
print(f"Date range: {df_raw.index.min()} -> {df_raw.index.max()}")

In [ ]:
# 3. Feature engineering: OHLC + date parts + cyclical encoding + Year MinMax

df = df_raw.copy()
df = df[['Open','High','Low','Close']]

df['Year'] = df.index.year
df['Month'] = df.index.month
df['Day'] = df.index.day

# Cyclical encoding for Month and Day
from math import pi

df['Month_sin'] = np.sin(2*pi*df['Month']/12)
df['Month_cos'] = np.cos(2*pi*df['Month']/12)
df['Day_sin'] = np.sin(2*pi*df['Day']/31)
df['Day_cos'] = np.cos(2*pi*df['Day']/31)

# MinMax scale Year
yr_min, yr_max = df['Year'].min(), df['Year'].max()
df['Year_mm'] = (df['Year'] - yr_min) / (yr_max - yr_min + 1e-8)

target_col = 'Close'
feature_cols = ['Open','High','Low','Close','Year_mm','Month_sin','Month_cos','Day_sin','Day_cos']

print(df[feature_cols].head())

In [ ]:
# 4. Stationarity diagnostics (ACF/PACF, differencing, ADF)
close_series = df[target_col]
fig, axes = plt.subplots(2,2, figsize=(12,8))
plot_acf(close_series.dropna(), ax=axes[0,0], lags=60)
axes[0,0].set_title('ACF - raw')
plot_pacf(close_series.dropna(), ax=axes[0,1], lags=60, method='ywm')
axes[0,1].set_title('PACF - raw')

close_diff = close_series.diff().dropna()
plot_acf(close_diff, ax=axes[1,0], lags=60)
axes[1,0].set_title('ACF - diff')
plot_pacf(close_diff, ax=axes[1,1], lags=60, method='ywm')
axes[1,1].set_title('PACF - diff')
plt.tight_layout(); plt.show()

adf_raw_p = adfuller(close_series.dropna())[1]
adf_diff_p = adfuller(close_diff)[1]
print(f"ADF p-value raw: {adf_raw_p:.4f}; differenced: {adf_diff_p:.4f}")

# Choose window L and horizon H based on slow ACF decay -> pick L=60, H=1
L = 60
H = 1
print(f"Using lookback L={L}, horizon H={H}")

In [ ]:
# 5. Window builder

def build_windows(data: pd.DataFrame, feature_cols: List[str], target_col: str, L: int, H: int):
    X_list, y_list, idx_list = [], [], []
    vals = data[feature_cols + [target_col]].values
    for i in range(L, len(vals)-H+1):
        X_list.append(vals[i-L:i, :len(feature_cols)])
        y_list.append(vals[i:i+H, -1])
        idx_list.append(data.index[i])
    return np.array(X_list), np.array(y_list), idx_list

X_all, y_all, idx_all = build_windows(df, feature_cols, target_col, L, H)
print(f"Windows: X {X_all.shape}, y {y_all.shape}")

In [ ]:
# 6. Chronological split 70/20/10 with explicit 2024–2025 test coverage
idx_series = pd.to_datetime(idx_all)

def split_by_time(X, y, idx, test_start=pd.Timestamp('2024-01-01')):
    mask_test = idx >= test_start
    X_test, y_test = X[mask_test], y[mask_test]
    idx_test = idx[mask_test]
    X_rem, y_rem, idx_rem = X[~mask_test], y[~mask_test], idx[~mask_test]
    n = len(X_rem)
    n_train = int(0.7*n)
    n_val = int(0.2*n)
    X_train, y_train, idx_train = X_rem[:n_train], y_rem[:n_train], idx_rem[:n_train]
    X_val, y_val, idx_val = X_rem[n_train:n_train+n_val], y_rem[n_train:n_train+n_val], idx_rem[n_train:n_train+n_val]
    return (X_train, y_train, idx_train), (X_val, y_val, idx_val), (X_test, y_test, idx_test)

(train_split, val_split, test_split) = split_by_time(X_all, y_all, np.array(idx_all))
X_train, y_train, idx_train = train_split
X_val, y_val, idx_val = val_split
X_test, y_test, idx_test = test_split

assert idx_train.max() < idx_val.min() < idx_test.min()
print(f"Train/Val/Test sizes: {len(X_train)}/{len(X_val)}/{len(X_test)}")
print(f"Train last date {idx_train.max()}, Val start {idx_val.min()}, Test start {idx_test.min()}")

In [ ]:
# 7. Z-score normalization (fit on train only)
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0) + 1e-8

X_train_n = (X_train - mu) / sigma
X_val_n = (X_val - mu) / sigma
X_test_n = (X_test - mu) / sigma

# Histogram check on a couple of features
plt.figure(figsize=(10,4))
plt.hist(X_train_n[:,:,0].flatten(), bins=50, alpha=0.6, label='feat0')
plt.hist(X_train_n[:,:,1].flatten(), bins=50, alpha=0.6, label='feat1')
plt.legend(); plt.title('Normalized feature histograms'); plt.show()

In [ ]:
# 8. Dataset and DataLoader
class WindowedTimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = WindowedTimeSeriesDataset(X_train_n, y_train)
val_ds = WindowedTimeSeriesDataset(X_val_n, y_val)
test_ds = WindowedTimeSeriesDataset(X_test_n, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=False)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print(next(iter(train_loader))[0].shape)

In [ ]:
# 9. Metrics utilities (RMSE/MAE/MAPE/R2)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def mape(y_true, y_pred):
    eps = 1e-8
    return float(np.mean(np.abs((y_true - y_pred) / (y_true + eps))))

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2) + 1e-8
    return float(1 - ss_res / ss_tot)

# unit check
_test_y = np.array([1.,2.,3.])
print("Metrics sanity:", rmse(_test_y,_test_y), mae(_test_y,_test_y), mape(_test_y,_test_y), r2(_test_y,_test_y))

In [ ]:
# 10. Models from src/models/sp_models.py
from models.sp_models import MLPRegressor, ConvLSTMRegressor, ConvGRURegressor, TCNRegressor

# flatten helper
input_dim = X_train_n.shape[1]*X_train_n.shape[2]

def count_params(model):
    return sum(p.numel() for p in model.parameters())

mlp = MLPRegressor(input_dim=input_dim, hidden_dim=128, dropout=0.2).to(device)
print("MLP params", count_params(mlp))

cnn_lstm = ConvLSTMRegressor(in_channels=X_train_n.shape[2], cnn_hidden=64, lstm_hidden=128, out_dim=H).to(device)
print("CNN+LSTM params", count_params(cnn_lstm))

cnn_gru = ConvGRURegressor(in_channels=X_train_n.shape[2], cnn_hidden=64, gru_hidden=128, out_dim=H).to(device)
print("CNN+GRU params", count_params(cnn_gru))

tcn = TCNRegressor(in_channels=X_train_n.shape[2], channels=48, kernel_size=3, num_blocks=3).to(device)
print("TCN params", count_params(tcn))

In [ ]:
# 11. Training/eval loop (shared)

def train_model(model, loaders, epochs=20, lr=1e-3, weight_decay=1e-4, ckpt_path="model.pt"):
    train_loader, val_loader = loaders
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    best = float('inf')
    history = {"train": [], "val": []}
    start = time.time()
    for ep in range(epochs):
        model.train()
        tr_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            if xb.dim() == 3 and isinstance(model, MLPRegressor):
                xb = xb.reshape(xb.size(0), -1)
            out = model(xb)
            loss = criterion(out, yb.squeeze())
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tr_loss += loss.item()
        tr_loss /= len(train_loader)
        model.eval(); va_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                if xb.dim() == 3 and isinstance(model, MLPRegressor):
                    xb = xb.reshape(xb.size(0), -1)
                out = model(xb)
                loss = criterion(out, yb.squeeze())
                va_loss += loss.item()
        va_loss /= len(val_loader)
        history["train"].append(tr_loss); history["val"].append(va_loss)
        print(f"Epoch {ep+1}: train {tr_loss:.4f} val {va_loss:.4f}")
        if va_loss < best:
            best = va_loss
            torch.save(model.state_dict(), ckpt_path)
    duration = time.time() - start
    print(f"Best val {best:.4f} in {duration/60:.1f} min")
    return history


def evaluate_model(model, loader):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            if xb.dim() == 3 and isinstance(model, MLPRegressor):
                xb = xb.reshape(xb.size(0), -1)
            out = model(xb).cpu().numpy().squeeze()
            preds.append(out)
            trues.append(yb.numpy().squeeze())
    y_true = np.concatenate(trues)
    y_pred = np.concatenate(preds)
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": mae(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
        "R2": r2(y_true, y_pred),
        "y_true": y_true,
        "y_pred": y_pred,
    }


In [ ]:
# 12. Train baseline MLP
hist_mlp = train_model(mlp, (train_loader, val_loader), epochs=20, lr=1e-3, weight_decay=1e-4, ckpt_path='mlp_best.pt')

plt.figure(figsize=(8,4))
plt.plot(hist_mlp['train'], label='train'); plt.plot(hist_mlp['val'], label='val'); plt.legend(); plt.title('MLP Loss'); plt.grid(True); plt.show()

mlp.load_state_dict(torch.load('mlp_best.pt', map_location=device))
res_mlp = evaluate_model(mlp, test_loader)
print(res_mlp)

# Plot predictions vs actual for 2024-2025
start_mask = np.array(idx_test) >= pd.Timestamp('2024-01-01')
plt.figure(figsize=(10,4))
plt.plot(idx_test[start_mask], res_mlp['y_true'][start_mask], label='Actual')
plt.plot(idx_test[start_mask], res_mlp['y_pred'][start_mask], label='MLP Pred')
plt.title('MLP Predictions 2024-2025'); plt.legend(); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
# 13-15. Train CNN+LSTM and CNN+GRU (same protocol)

hist_lstm = train_model(cnn_lstm, (train_loader, val_loader), epochs=20, lr=1e-3, weight_decay=1e-4, ckpt_path='lstm_best.pt')
hist_gru = train_model(cnn_gru, (train_loader, val_loader), epochs=20, lr=1e-3, weight_decay=1e-4, ckpt_path='gru_best.pt')

plt.figure(figsize=(10,4))
plt.plot(hist_lstm['val'], label='LSTM val'); plt.plot(hist_gru['val'], label='GRU val'); plt.axhline(0.02, color='r', linestyle='--', label='target 0.02');
plt.legend(); plt.title('Val loss'); plt.grid(True); plt.show()

cnn_lstm.load_state_dict(torch.load('lstm_best.pt', map_location=device))
cnn_gru.load_state_dict(torch.load('gru_best.pt', map_location=device))
res_lstm = evaluate_model(cnn_lstm, test_loader)
res_gru = evaluate_model(cnn_gru, test_loader)
print('LSTM', res_lstm)
print('GRU', res_gru)

In [ ]:
# 16. Unified evaluation + comparison table
res_mlp_short = {k:v for k,v in res_mlp.items() if isinstance(v,float)}
res_lstm_short = {k:v for k,v in res_lstm.items() if isinstance(v,float)}
res_gru_short = {k:v for k,v in res_gru.items() if isinstance(v,float)}

results_df = pd.DataFrame([
    {"Model":"MLP","Params":count_params(mlp), **res_mlp_short},
    {"Model":"CNN+LSTM","Params":count_params(cnn_lstm), **res_lstm_short},
    {"Model":"CNN+GRU","Params":count_params(cnn_gru), **res_gru_short},
])
print(results_df)
results_df.to_csv('comparison.csv', index=False)

# 18. Overlay plot
plt.figure(figsize=(10,4))
plt.plot(idx_test, res_mlp['y_true'], label='Actual')
plt.plot(idx_test, res_mlp['y_pred'], label='MLP')
plt.plot(idx_test, res_lstm['y_pred'], label='LSTM')
plt.plot(idx_test, res_gru['y_pred'], label='GRU')
plt.legend(); plt.title('Predictions overlay (2024-2025)'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
# 17. Params + approx FLOPs (rough) + timing already logged

# 19. Recursive multi-step forecasting into 2026–2027 using best GRU
cnn_gru.load_state_dict(torch.load('gru_best.pt', map_location=device))
cnn_gru.eval()

horizon_days = 365 * 2
window = X_test_n[-1:].copy()  # last normalized window
preds_future = []
for _ in range(horizon_days):
    xb = torch.tensor(window, dtype=torch.float32, device=device)
    out = cnn_gru(xb).detach().cpu().numpy().squeeze()
    preds_future.append(out)
    # roll window: drop first step, append new prediction
    # copy last timestep and update Close feature with prediction
    new_feat = window[:, -1:, :].copy()
    new_feat[0, 0, 3] = out  # update Close feature (index 3)
    window = np.concatenate([window[:, 1:, :], new_feat], axis=1)

future_index = pd.date_range(start=idx_test[-1]+pd.Timedelta(days=1), periods=horizon_days, freq='D')
plt.figure(figsize=(10,4))
plt.plot(future_index, preds_future, label='GRU recursive')
plt.title('Recursive forecast 2026-2027'); plt.legend(); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# 20. Bonus: CNN-only (TCN)
hist_tcn = train_model(tcn, (train_loader, val_loader), epochs=15, lr=1e-3, weight_decay=1e-4, ckpt_path='tcn_best.pt')
tcn.load_state_dict(torch.load('tcn_best.pt', map_location=device))
res_tcn = evaluate_model(tcn, test_loader)
print('TCN', res_tcn)

# Add to comparison
res_tcn_short = {k:v for k,v in res_tcn.items() if isinstance(v,float)}
results_df = pd.concat([results_df, pd.DataFrame([{**{"Model":"TCN","Params":count_params(tcn)}, **res_tcn_short}])], ignore_index=True)
print(results_df)

# Overlay including TCN
plt.figure(figsize=(10,4))
plt.plot(idx_test, res_mlp['y_true'], label='Actual')
plt.plot(idx_test, res_mlp['y_pred'], label='MLP')
plt.plot(idx_test, res_lstm['y_pred'], label='LSTM')
plt.plot(idx_test, res_gru['y_pred'], label='GRU')
plt.plot(idx_test, res_tcn['y_pred'], label='TCN')
plt.legend(); plt.title('Overlay with TCN'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# 21. Summary and Conclusions
print("## CA4 Question 2 - S&P 500 Forecasting Summary")
print("\n### Key Findings:")
print(f"- Best performing model: {results_df.loc[results_df['RMSE'].idxmin(), 'Model']} (RMSE: {results_df['RMSE'].min():.4f})")
print(f"- Stationarity: Raw series is non-stationary (ADF p={adf_raw_p:.4f}), differenced series is stationary (ADF p={adf_diff_p:.4f})")
print(f"- Window size L={L}, Horizon H={H} chosen based on ACF decay")
print(f"- Chronological split ensures no leakage: Train {idx_train.max()}, Val {idx_val.min()}, Test {idx_test.min()}")
print("\n### Model Comparison:")
print(results_df.to_string(index=False))
print("\n### Recursive Forecasting:")
print(f"- Extended GRU predictions {horizon_days} days into 2026-2027")
print("- Note: Recursive forecasting accumulates errors over time")
